# **Assignment 3 — Simplex Algorithm (No Initial Feasible Point Given)**
### **Piyush Anand** • **Roll No:** CS25MTECH12009

**Input (`Testcase.csv`):**  
- **Row 1 (excluding last column):** Cost vector **c** of length *n*  
- **Rows 2 … m+1, columns 1 … n:** Constraint matrix **A** of size *(m × n)*  
- **Last column in rows 2 … m+1:** Right-hand-side vector **b** of length *m*

---

**Objective:**  
Maximize  $ c^T x $  subject to  $ A x \le b $.

---

**Expected Output:**  
- Display the **initial feasible point** (computed using Phase I)  
- Print the **sequence of vertices** visited and the **objective value** at each vertex  

---

**Method Overview:**  

1. **Phase I (Auxiliary Linear Program):**  
   - Introduce an auxiliary variable $t \ge 0$  
   - Solve the modified system:  

     $$
     A x \le b + t \mathbf{1}
     $$  

   - **Objective:** Minimize $t$ (equivalently, maximize $-t$)  
   - The resulting feasible point becomes the starting vertex for Phase II.  

2. **Phase II (Geometric Simplex):**  
   - Begin from the feasible vertex obtained in Phase I.  
   - Apply the **geometric simplex algorithm** to move along edges of the feasible region, improving the objective value until optimality or unboundedness is reached.  

---

**Degeneracy Handling:**  
If the current point has **more than n tight constraints**, select **any n linearly independent tight constraints** to form the basis $A_B$.  
If the selected $A_B$ is **singular**, reselect another independent subset to ensure full rank.


In [ ]:
import numpy as np
import pandas as pd
from scipy.linalg import null_space



##  Read `Testcase.csv` (Assignment format)
- Row 1 (excluding last col) → `c`
- Rows 2..m+1, cols 1..n → `A`
- Last column (rows 2..m+1) → `b`


In [ ]:
def read_input_assignment3(filename):
    data = pd.read_csv(filename, header=None).values.astype(float)
    c = data[0, :-1]
    A = data[1:, :-1]
    b = data[1:, -1]
    m, n = A.shape
    return A, b, c, m, n


## Helper Functions
- `find_tight_rows`: which constraints are active at a point
- `pick_full_rank_subset`: pick **any n** independent rows from a set
- `basis_from_tight`: build a basis index list `B` from the current tight set


In [ ]:
def find_tight_rows(A, x, b, tol=1e-9):
    """Return (tight_indices, A_tight, A_loose)."""
    r = A @ x - b
    tight_mask = np.abs(r) < tol
    tight_idx = np.where(tight_mask)[0]
    return tight_idx, A[tight_mask], A[~tight_mask]

def pick_full_rank_subset(A_rows, want_cols):
    """
    Greedy: pick any 'want_cols' independent rows from A_rows (shape k×n).
    Return row indices local to A_rows, or None if not found.
    """
    idxs = []
    for i in range(A_rows.shape[0]):
        trial = idxs + [i]
        if np.linalg.matrix_rank(A_rows[trial, :]) == len(trial):
            idxs = trial
            if len(idxs) == want_cols:
                return idxs
    return None

def basis_from_tight(A, tight_idx, n):
    """
    From the global 'A' and the list of global tight indices, pick any 'n'
    linearly independent constraints to form a basis. Returns the global indices
    of the chosen basis or None.
    """
    if len(tight_idx) < n:
        return None
    A_tight_full = A[tight_idx, :]
    local = pick_full_rank_subset(A_tight_full, n)
    if local is None:
        return None
    return [int(tight_idx[i]) for i in local]


## **Phase I (Auxiliary Linear Program)**

We augment variables as  

$$
y =
\begin{bmatrix}
x \\
t
\end{bmatrix}
\in \mathbb{R}^{\,n+1}
$$

with constraints:

$$
A x \le b + t \cdot \mathbf{1}
\quad \Longleftrightarrow \quad
[A \;|\; -\mathbf{1}]\,y \le b,
\qquad t \ge 0
$$

---

### **Objective Function**

Minimize $t$  ⇔  **maximize** $-t$ (to keep the problem in maximization form).

---

### **Feasible Starting Point**

Take  

$$
x = 0, \qquad
t_0 = \max(0,\,-\min(b))
$$

so that  

$$
0 \le b + t_0.
$$

This guarantees a feasible starting point.  
From this point, we **move to a vertex** in the augmented space and apply the **geometric simplex algorithm** to reach feasibility (i.e., $t^{*}=0$).


In [ ]:
def build_phase1_lp(A, b):
    """
    Build auxiliary LP in variables y = [x; t] with constraints:
        [A | -1] y <= b
        [0 | -1] y <= 0      (i.e., t >= 0)
    Maximize -t  (minimize t).
    Returns (A_hat, b_hat, c_hat, y0) with a feasible starting point y0.
    """
    m, n = A.shape

    # Ax <= b + t*1  <=>  [A | -1] [x; t] <= b
    A1 = np.hstack([A, -np.ones((m, 1))])
    b1 = b.copy()

    # t >= 0  <=>  -t <= 0
    A2 = np.hstack([np.zeros((1, n)), -np.ones((1, 1))])
    b2 = np.zeros(1)

    A_hat = np.vstack([A1, A2])
    b_hat = np.concatenate([b1, b2])

    # Phase I objective: maximize -t
    c_hat = np.zeros(n + 1)
    c_hat[-1] = -1.0

    # Feasible start: x = 0, pick t0 = max(0, max_i{-b_i}) = max(0, -min(b))
    t0 = max(0.0, float(-np.min(b)))
    y0 = np.zeros(n + 1)
    y0[-1] = t0

    return A_hat, b_hat, c_hat, y0


## Feasible point → Vertex
Follow a null-space direction of the currently tight constraints and perform a ratio test
to hit the next facet. Repeat until `rank(A_tight) = n_dim`.
When more than `n_dim` constraints are tight, we **pick any n_dim independent rows**.


In [ ]:
from scipy.linalg import null_space
import numpy as np

def move_feasible_to_vertex(A, b, x, n_dim, tol=1e-9, max_hops=10000):
    """
    Start from a FEASIBLE point x in R^{n_dim} and move to a vertex (rank(A_tight)=n_dim).
    Handles degeneracy by selecting any n_dim independent tight rows.
    Tries both u and -u when doing the ratio test.
    Returns (vertex, path).
    """
    path = [x.copy()]
    for _ in range(max_hops):
        # Identify tight/loose sets
        r = A @ x - b
        tight_mask = np.abs(r) < tol
        tight_idx = np.where(tight_mask)[0]
        A_tight = A[tight_idx, :]

        # If too many tight rows, keep any n_dim independent subset as the working "tight"
        if A_tight.shape[0] > 0:
            if A_tight.shape[0] > n_dim:
                # pick any n_dim independent among the tight
                def pick_full_rank_subset(A_rows, want_cols):
                    idxs = []
                    for i in range(A_rows.shape[0]):
                        trial = idxs + [i]
                        if np.linalg.matrix_rank(A_rows[trial, :]) == len(trial):
                            idxs = trial
                            if len(idxs) == want_cols:
                                return idxs
                    return None
                local = pick_full_rank_subset(A_tight, n_dim)
                if local is not None:
                    A_tight = A_tight[local, :]

        # Check rank
        rank = np.linalg.matrix_rank(A_tight) if A_tight.size else 0
        if rank == n_dim:
            return x, path  # at a vertex

        # Direction from null-space of A_tight
        if A_tight.size == 0:
            u = np.random.randn(n_dim)
        else:
            ns = null_space(A_tight)
            if ns.size == 0:
                # fallback small random nudge
                u = np.random.randn(n_dim)
            else:
                u = ns[:, 0]

        # Ratio test on loose constraints. Try u; if it fails, try -u.
        def step_along(direction):
            loose_idx = np.where(~tight_mask)[0]
            alphas = []
            for j in loose_idx:
                aj = A[j, :]
                denom = float(aj @ direction)
                if denom > tol:
                    t = (b[j] - float(aj @ x)) / denom
                    if t > tol:
                        alphas.append(t)
            return min(alphas) if alphas else None

        alpha = step_along(u)
        if alpha is None:
            alpha = step_along(-u)
            if alpha is not None:
                u = -u

        if alpha is None:
            # cannot find a blocking constraint => unbounded along both ±u relative to the current polyhedron
            return x, path

        x = x + alpha * u
        path.append(x.copy())

    return x, path


##   Simplex (Vertex-to-Vertex)
- Build a basis `B` from **any n independent tight rows**.
- Directions are columns of `-A_B^{-1}`.
- Choose direction with largest `cᵀv > 0`.
- Ratio test to the nearest facet; pivot entering/leaving constraints.
- If the chosen basis is singular, **reselect** an independent subset.


In [ ]:
import numpy as np

def geometric_simplex_trace(A, b, c, x0, tol=1e-9):
    """
    Phase II: Vertex-to-vertex geometric simplex algorithm with correct tight/untight handling.
    Key changes:
      - Ratio test includes *all* non-basis constraints, including currently tight ones.
      - Allow zero-step (t* = 0) pivots to handle degeneracy (do not step past a tight facet).
      - Rebuild basis from truly tight constraints when needed.
    Always returns (x_star, obj_star, vertex_path).
    """
    m, n = A.shape

    # Build initial basis from tight constraints (pick any n independent)
    def find_tight_indices(x):
        r = A @ x - b
        return np.where(np.abs(r) < tol)[0]

    def pick_full_rank_subset_from(indices):
        if len(indices) < n:
            return None
        A_t = A[indices, :]
        chosen = []
        for i_local, i_global in enumerate(indices):
            trial = chosen + [i_local]
            if np.linalg.matrix_rank(A_t[trial, :]) == len(trial):
                chosen = trial
                if len(chosen) == n:
                    break
        if len(chosen) < n:
            return None
        return [int(indices[k]) for k in chosen]

    tight_idx = find_tight_indices(x0)
    B = pick_full_rank_subset_from(list(tight_idx))
    if B is None:
        print("Error: initial point not on enough independent active constraints (not a vertex).")
        return x0, float(c @ x0), [x0.copy()]

    vertex_path = [x0.copy()]
    iteration = 0

    print("Iter | Vertex (x)                       | Obj         | Direction info")
    print("---------------------------------------------------------------------------")

    while True:
        iteration += 1

        # Build A_B and its inverse safely
        A_B = A[B, :]
        if np.linalg.matrix_rank(A_B) < n:
            # Try to rebuild from tight constraints at current x0
            tight_idx = find_tight_indices(x0)
            B_new = pick_full_rank_subset_from(list(tight_idx))
            if B_new is None:
                print("Could not build a full-rank basis at this vertex; stopping.")
                return x0, float(c @ x0), vertex_path
            B = B_new
            A_B = A[B, :]

        try:
            V = -np.linalg.inv(A_B)   # directions are columns of -A_B^{-1}
        except np.linalg.LinAlgError:
            # Rebuild once from tight set
            tight_idx = find_tight_indices(x0)
            B_new = pick_full_rank_subset_from(list(tight_idx))
            if B_new is None:
                print("Singular basis and cannot rebuild; stopping.")
                return x0, float(c @ x0), vertex_path
            B = B_new
            V = -np.linalg.inv(A[B, :])

        dirs = [V[:, j] for j in range(n)]
        gains = [float(c @ v) for v in dirs]

        print(f"{iteration:4d} | {np.round(x0,6)} | {float(c @ x0):11.6f} | ", end="")

        # Optimality: no direction improves
        if all(g <= tol for g in gains):
            print("\nReached optimal vertex.")
            print(f"x* = {np.round(x0,6)}, Objective = {float(c @ x0):.6f}")
            return x0, float(c @ x0), vertex_path

        # Choose entering direction (largest positive gain)
        i_enter = int(np.argmax(gains))
        if gains[i_enter] <= tol:
            print("\nNo improving direction (within tolerance).")
            return x0, float(c @ x0), vertex_path

        v = dirs[i_enter]
        print(f"dir {i_enter}, cᵀv = {gains[i_enter]:.6f}")

        # Ratio test over ALL non-basis constraints (including currently tight ones)
        N_all = [i for i in range(m) if i not in B]
        blockers = []
        for j in N_all:
            aj = A[j, :]
            denom = float(aj @ v)
            if denom > tol:
                num = b[j] - float(aj @ x0)
                # Accept zero-step blockers: t >= 0 (allow degenerate pivot)
                if num >= -tol:
                    t = max(0.0, num / denom)  # clip tiny negative to 0
                    blockers.append((t, j))

        if not blockers:
            print("   → Unbounded in this direction.")
            return x0, np.inf, vertex_path

        # Choose nearest blocker; if t* = 0 we will do a degenerate pivot (no movement)
        t_star, j_enter = min(blockers, key=lambda z: z[0])

        if t_star > tol:
            # Proper movement
            x0 = x0 + t_star * v
            vertex_path.append(x0.copy())
        else:
            # Degenerate pivot: stay at same x0, just change basis
            pass

        # Recompute tight set at the (possibly unchanged) x0
        tight_idx = find_tight_indices(x0)
        # Leaving constraint is the current basis element aligned with direction
        j_leave = B[i_enter]

        # If the entering constraint is already in the basis (can happen in degeneracy), pick an alternative
        if j_enter in B:
            # Try to rebuild a basis from current tight set
            B_new = pick_full_rank_subset_from(list(tight_idx))
            if B_new is None:
                # Fall back to classic pivot: keep B but skip update this round
                print("   (Degenerate tie; basis unchanged this round)")
            else:
                B = B_new
        else:
            # Standard pivot
            B = B.copy()
            B[i_enter] = j_enter

        print(f"   → Move t*={t_star:.6f} → New vertex {np.round(x0,6)}")
        print(f"     Entering constraint {j_enter}, Leaving constraint {j_leave}")
        print("---------------------------------------------------------------------------")


## Full pipeline (Phase I + Phase II)
1. Build Phase I LP `[A | -1] y ≤ b`, maximize `-t`; start from feasible `y0 = [0,…,0,t0]`.
2. **Move feasible→vertex** in augmented space, then run geometric simplex on Phase I.
3. If Phase I optimum `t* > 0` ⇒ **infeasible** original LP.
4. If `t* = 0`, take `x0 = y*[:-1]`. If needed, **move feasible→vertex** in original space.
5. Run geometric simplex on original LP; print initial feasible point and the full trace.


In [ ]:
def simplex_full_no_initial_point(csv_filename, tol=1e-9):
    # Read original LP
    A, b, c, m, n = read_input_assignment3(csv_filename)

    # Phase I: build auxiliary LP and get a feasible start
    A_hat, b_hat, c_hat, y0 = build_phase1_lp(A, b)
    print("\n=== Phase I (Auxiliary LP — minimize t) ===")

    # Move feasible point -> vertex in augmented space (n+1 variables)
    y_vertex, _ = move_feasible_to_vertex(A_hat, b_hat, y0, n_dim=n+1, tol=tol)

    # Run geometric simplex (your Phase II engine) on Phase I from that vertex
    y_star, obj_aux, _ = geometric_simplex_trace(A_hat, b_hat, c_hat, y_vertex, tol=tol)

    t_star = float(y_star[-1])
    x_feas = y_star[:-1].copy()
    print(f"\nPhase I result: t* = {t_star:.6f}")
    if t_star > 1e-9:
        print("Original LP is infeasible (no feasible point).")
        return None, None

    print(f"Initial feasible point: x0 = {np.round(x_feas,8)}")

    # Ensure x0 is a vertex in original LP; if not, move feasible->vertex
    x_vertex, _ = move_feasible_to_vertex(A, b, x_feas, n_dim=n, tol=tol)

    print("\n=== Phase II (Geometric Simplex on original LP) ===")
    x_opt, obj_opt, visited = geometric_simplex_trace(A, b, c, x_vertex, tol=tol)

    print("\nSequence of vertices (Phase II):")
    print("-----------------------------------------------------------")
    print(f"Start (reported feasible): x = {np.round(x_feas,8)},  f = {float(c @ x_feas):.6f}")
    for k, (xk, fk) in enumerate(visited, 1):
        print(f"{k:2d}. x = {np.round(xk,8)},  f = {fk:.6f}")
    print(f"\nFinal: x* = {np.round(x_opt,8)},  f* = {obj_opt:.6f}")

    return x_opt, obj_opt


In [ ]:
# c
# rows: A | b
example = [
    [3, 2, ""],         # c
    [ 2,  1,  10],      # 2x1 + x2 <= 10
    [-1,  2,   4],      # -x1 + 2x2 <= 4
    [ 1,  0,  -1],      #  x1 <= -1     (negative b; allows x1 negative)
    [-1,  0,   5],      # -x1 <= 5  -> x1 >= -5
    [ 0, -1,   0],      # -x2 <= 0  -> x2 >= 0
]
pd.DataFrame(example).to_csv("Testcase.csv", header=False, index=False)
print("Wrote Testcase.csv (negative b, feasible):\n")
with open("Testcase.csv") as f:
    print(f.read())

_ = simplex_full_no_initial_point("Testcase.csv")


Wrote Testcase.csv (negative b, feasible):

3,2,
2,1,10
-1,2,4
1,0,-1
-1,0,5
0,-1,0


=== Phase I (Auxiliary LP — minimize t) ===
Iter | Vertex (x)                       | Obj         | Direction info
---------------------------------------------------------------------------
   1 | [4.25 6.75 5.25] |   -5.250000 | dir 0, cᵀv = 0.500000
   → Move t*=10.500000 → New vertex [-1.   1.5  0. ]
     Entering constraint 5, Leaving constraint 0
---------------------------------------------------------------------------
   2 | [-1.   1.5  0. ] |    0.000000 | 
Reached optimal vertex.
x* = [-1.   1.5  0. ], Objective = 0.000000

Phase I result: t* = 0.000000
Initial feasible point: x0 = [-1.   1.5]

=== Phase II (Geometric Simplex on original LP) ===
Iter | Vertex (x)                       | Obj         | Direction info
---------------------------------------------------------------------------
   1 | [-1.   1.5] |   -0.000000 | 
Reached optimal vertex.
x* = [-1.   1.5], Objective = -0.000000

S

In [ ]:
_ = simplex_full_no_initial_point("testcase6.csv")



=== Phase I (Auxiliary LP — minimize t) ===
Iter | Vertex (x)                       | Obj         | Direction info
---------------------------------------------------------------------------
   1 | [ 1. -1.  0.] |    0.000000 | 
Reached optimal vertex.
x* = [ 1. -1.  0.], Objective = 0.000000

Phase I result: t* = 0.000000
Initial feasible point: x0 = [ 1. -1.]

=== Phase II (Geometric Simplex on original LP) ===
Iter | Vertex (x)                       | Obj         | Direction info
---------------------------------------------------------------------------
   1 | [ 1. -1.] |    3.000000 | 
Reached optimal vertex.
x* = [ 1. -1.], Objective = 3.000000

Sequence of vertices (Phase II):
-----------------------------------------------------------
Start (reported feasible): x = [ 1. -1.],  f = 3.000000
 1. x = 1.0,  f = -1.000000

Final: x* = [ 1. -1.],  f* = 3.000000
